# Séance 2 — Chargement, inspection, filtrage

**Analyse des données — L3 Économie**

Nous travaillons sur `cm02-communes.csv` : quarante-cinq communes françaises, une ligne par commune.

**Ouvrez le dictionnaire des variables avant de commencer.** Il est déposé sur le site du cours, à côté du fichier. Sans lui, ce qui suit n'a pas de sens.

> Ces données sont fabriquées pour l'enseignement. Les ordres de grandeur sont plausibles, les valeurs ne sont pas celles de l'INSEE.

## 1. Les bibliothèques

In [ ]:
import pandas as pd
import numpy as np

## 2. Charger — première tentative

Chargement par défaut, sans rien préciser.

In [ ]:
URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/cm02-communes.csv")

# Cette cellule produit une ERREUR. C'est voulu.
df = pd.read_csv(URL)

`UnicodeDecodeError`. Le fichier n'est pas encodé en UTF-8.

Les fichiers produits par les administrations françaises utilisent souvent `latin-1`, un encodage plus ancien. Il faut le déclarer.

In [ ]:
df = pd.read_csv(URL, encoding="latin-1")
print(df.shape)
df.head()

**Une seule colonne.** Le séparateur n'est pas la virgule.

Les fichiers français utilisent le point-virgule, précisément parce que la virgule sert de séparateur décimal.

In [ ]:
df = pd.read_csv(URL, encoding="latin-1", sep=";")
print(df.shape)
df.head()

Sept colonnes, quarante-cinq lignes. Le fichier est lu.

Il reste faux.

## 3. Inspecter avant de faire quoi que ce soit

Quatre commandes, systématiquement, avant tout calcul.

In [ ]:
print(df.shape)              # dimensions
print(df.columns.tolist())   # noms de variables
print()
df.info()                    # types et valeurs non manquantes

Trois anomalies dans cette sortie. Cherchez-les avant de continuer.

1. `CODGEO` est un entier — or c'est un identifiant.
2. `POP` et `REVENU_MEDIAN` sont du texte — or ce sont des nombres.
3. `TAUX_CHOMAGE` est du texte lui aussi.

In [ ]:
df["CODGEO"].head(3)

Le code de Bourg-en-Bresse est `01053`. Il est devenu `1053`.

Le zéro initial a disparu, et avec lui la correspondance avec tout autre fichier communal. La fusion échouera en séance 4 — sans message d'erreur, en appariant simplement moins de lignes que prévu.

**Un identifiant n'est jamais un nombre.**

## 4. Charger correctement

In [ ]:
df = pd.read_csv(
    URL,
    encoding="latin-1",
    sep=";",
    decimal=",",                                    # virgule decimale
    thousands=" ",                                  # espace pour les milliers
    dtype={"CODGEO": str, "CAT_URBAINE": str}       # les identifiants en texte
)

df.info()

In [ ]:
df.head()

`POP` et `REVENU_MEDIAN` sont maintenant numériques, et `CODGEO` a retrouvé son zéro.

Mais `TAUX_CHOMAGE` reste du texte. Pourquoi ?

## 5. Le cas du taux de chômage

In [ ]:
df["TAUX_CHOMAGE"].value_counts().head(8)

La valeur `nd` apparaît six fois. Le dictionnaire dit ce qu'elle signifie : **donnée non diffusée**, pour les communes de moins de 2 000 habitants, au titre du secret statistique.

Une seule modalité de texte suffit à empêcher pandas de lire toute la colonne comme numérique — et l'option `decimal=","` ne s'applique donc pas à elle.

In [ ]:
# Premiere tentative : convertir directement
essai = pd.to_numeric(df["TAUX_CHOMAGE"], errors="coerce")
print("valeurs manquantes apres conversion :", essai.isna().sum(), "sur", len(df))

**Tout est perdu.** Les virgules décimales n'ont pas été traduites, donc aucune valeur n'est convertible.

Il faut remplacer la virgule par un point, puis convertir.

In [ ]:
df["TAUX_CHOMAGE"] = pd.to_numeric(
    df["TAUX_CHOMAGE"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("valeurs manquantes :", df["TAUX_CHOMAGE"].isna().sum())
print("moyenne :", round(df["TAUX_CHOMAGE"].mean(), 2), "%")

Six valeurs manquantes, qui correspondent aux six `nd`.

**Attention à ce que dit cette moyenne.** Les communes absentes sont toutes petites. Le 16,18 % calculé décrit les communes moyennes et grandes, pas l'ensemble. Nous y reviendrons en séance 3.

## 6. La non-réponse codée en chiffres

Le dictionnaire signale que `NB_ENTREPRISES` code la non-réponse par `9999`.

In [ ]:
print("moyenne brute :", round(df["NB_ENTREPRISES"].mean(), 1))
print("communes a 9999 :", (df["NB_ENTREPRISES"] == 9999).sum())

In [ ]:
df["NB_ENTREPRISES"] = df["NB_ENTREPRISES"].replace(9999, np.nan)
print("moyenne corrigee :", round(df["NB_ENTREPRISES"].mean(), 1))

Le calcul fonctionnait parfaitement avant la correction. Il était faux.

**Aucune erreur ne s'affiche jamais pour ce genre de problème.** Seul le dictionnaire vous prévient.

## 7. Sélectionner et filtrer

In [ ]:
df["POP"]                                  # une colonne : une Series
df[["LIBGEO", "POP", "TAUX_CHOMAGE"]].head()   # plusieurs : un DataFrame

Les doubles crochets ne sont pas une coquetterie : vous passez une **liste** de noms — la structure vue en séance 1.

In [ ]:
# value_counts : le premier reflexe sur une variable qualitative
df["CAT_URBAINE"].value_counts().sort_index()

Quatre modalités, comme annoncé par le dictionnaire. Si une cinquième apparaissait, ou si l'une manquait, il faudrait comprendre pourquoi avant d'aller plus loin.

In [ ]:
# Filtrer : la condition produit une suite de vrai/faux
grandes = df[df["POP"] > 100000]
print(len(grandes), "communes de plus de 100 000 habitants")
grandes[["LIBGEO", "POP"]].sort_values("POP", ascending=False)

In [ ]:
# Combiner : les parentheses sont OBLIGATOIRES
cible = df[(df["POP"] > 50000) & (df["TAUX_CHOMAGE"] > 18)]
cible[["LIBGEO", "POP", "TAUX_CHOMAGE"]]

Sans les parenthèses autour de chaque condition, Python évalue dans le mauvais ordre et renvoie une erreur peu explicite. Prenez l'habitude tout de suite.

## 8. Le piège de l'indexation chaînée

In [ ]:
# Ce que beaucoup ecrivent — et qui ne fonctionne pas
df[df["POP"] < 2000]["TAUX_CHOMAGE"] = 0

# La valeur a-t-elle change ?
print(df.loc[df["POP"] < 2000, "TAUX_CHOMAGE"].head(3))

Deux crochets successifs produisent une **copie**. La modification s'applique à la copie, puis disparaît. Aucune erreur, aucun effet.

**Pour modifier, c'est toujours `.loc`** — lignes et colonnes en une seule opération.

In [ ]:
# La bonne facon d'ecrire (ici on ne modifie rien, on selectionne)
df.loc[df["POP"] < 2000, ["LIBGEO", "POP", "TAUX_CHOMAGE"]]

## 9. Chaîner plutôt qu'empiler

In [ ]:
resultat = (
    df
    .loc[df["CAT_URBAINE"] == "1", ["LIBGEO", "POP", "REVENU_MEDIAN", "TAUX_CHOMAGE"]]
    .sort_values("TAUX_CHOMAGE", ascending=False)
)
resultat

Une opération par ligne, dans l'ordre où on les pense. Les parenthèses extérieures autorisent le retour à la ligne.

C'est l'idiome courant de pandas, et vous le retrouverez dans tout code écrit par un praticien.

## 10. Les trois contrôles avant de quitter un fichier

1. `df.shape` correspond-il à ce qu'annonce la documentation ?
2. `df.info()` : chaque colonne a-t-elle le type attendu ?
3. `value_counts()` sur les variables qualitatives : les modalités correspondent-elles au dictionnaire ?

Deux minutes. Elles vous éviteront la moitié des erreurs du semestre — et elles constituent une question type du contrôle continu.

## À faire avant la séance 3

1. Reprenez ce notebook et calculez le revenu médian moyen par catégorie urbaine. Le résultat vous surprend-il ?
2. Repérez, **sans les traiter**, toutes les variables contenant des valeurs manquantes.
3. Vérifiez que le notebook s'exécute de haut en bas.

`stefaniamarcassa.github.io/analyse_des_donnees`